# Topic modelling with LDA

LDA describes each article as a **mixture of topics** and each topic as a distribution over words. We choose the topic count, inspect the leading words, and read the articles before naming the topics.

Run the cells from top to bottom. This notebook and the BERTopic notebook download and sample the same BBC articles. The main route is: download → counts → fit four versions → compare coherence → select a model → reading → coherence → name topics and describe your DataFrame → category comparison. Stability and the interactive map are optional extensions at the end.

The code uses short loops so you can inspect each intermediate object. All outputs have been cleared for the new dataset.

In [ ]:
# Hide this known notebook-library warning; keep other warnings visible.
import warnings

warnings.filterwarnings(
    "ignore",
    message=r"datetime\.datetime\.utcnow\(\) is deprecated.*",
    category=DeprecationWarning,
    module=r"jupyter_client\.session",
)

### Install packages

In Colab, run the next cell before importing libraries. In a local notebook, uncomment it if packages are missing. If installation asks for a runtime restart, restart and continue from the download section. A CPU is sufficient; no GPU or Google Drive connection is needed.

In [ ]:
# %pip install "gensim>=4" "pyLDAvis>=3" pandas scikit-learn matplotlib

### After installation or a runtime restart

Run the warning filter below before continuing. It is repeated here so it also applies if you restarted after installing packages. It hides only this specific `jupyter_client` deprecation warning; other warnings remain visible.

**Already printed warnings remain in the output.** Running the filter again does not erase them. Clear the affected cell's output (or clear all outputs), then rerun the affected cells after this filter. After a runtime restart, rerun this filter and the exercise cells because the previous filter is no longer active.

In [ ]:
# Hide this known notebook-library warning; keep other warnings visible.
import warnings

warnings.filterwarnings(
    "ignore",
    message=r"datetime\.datetime\.utcnow\(\) is deprecated.*",
    category=DeprecationWarning,
    module=r"jupyter_client\.session",
)

## 1. Download and sample the BBC news dataset

We use the [BBC News dataset provided by Derek Greene and Pádraig Cunningham](https://github.com/derekgreene/bbc-datasets): 2,225 articles from 2004–2005, in business, entertainment, politics, sport and technology. The source provides it for non-commercial and research purposes; article copyright belongs to the BBC.

The next cell downloads a zip file and reads its text files directly. **No Google Drive mount, Canvas archive or manual upload is needed.** Internet access is needed for the download. Each row will contain an article ID, its original category, a title and the full text. We keep the category for a later check; the model will receive only text.

The `with` blocks close the download and zip when finished. `split("/")` reads the category from a path such as `bbc/sport/001.txt`. `decode` converts the file's bytes into text.

In [ ]:
from urllib.request import urlopen
from io import BytesIO
from zipfile import ZipFile
import pandas as pd

url = (
    "https://raw.githubusercontent.com/derekgreene/"
    "bbc-datasets/main/raw/bbc-fulltext.zip"
)

with urlopen(url, timeout=60) as response:
    downloaded_file = response.read()

rows = []
with ZipFile(BytesIO(downloaded_file)) as archive:
    for filename in sorted(archive.namelist()):
        parts = filename.split("/")
        if len(parts) == 3 and filename.endswith(".txt"):
            text = archive.read(filename).decode("utf-8-sig").strip()
            lines = text.splitlines()
            rows.append({
                "article_id": filename,
                "category": parts[1],
                "title": lines[0],
                "body": " ".join(lines[1:]),
                "full_text": text,
            })

articles = pd.DataFrame(rows)
print("Articles downloaded:", len(articles))

### Use the same 500 articles in both notebooks

Choose 100 articles from each category. A fixed seed makes the selection repeatable; sorting article IDs gives the same final order in both notebooks. Use the same settings in LDA and BERTopic.

For each article, we use its **title plus the first 100 words of its body**. This keeps the exercise manageable and reduces truncation by the sentence encoder. We are modelling article openings, not every paragraph. The full text remains available for reading.

The balanced sample makes broad subjects easier to compare, but it does not represent the original category proportions or all news coverage. Categories guide our sampling and later evaluation; they are never passed to either model.

**Keep encoder coverage consistent:** the BERTopic notebook checks whether any shared excerpt exceeds its encoder's token limit and stops if so. Reduce `OPENING_WORDS` in both notebooks and rerun both analyses if that happens. Keep the same sample and article order; do not shorten only BERTopic's text. The default MiniLM encoder reads all 100-word excerpts in this sample.

In [ ]:
SEED = 42
ARTICLES_PER_CATEGORY = 100
OPENING_WORDS = 100

samples = []
for category, group in articles.groupby("category"):
    chosen = group.sample(n=ARTICLES_PER_CATEGORY, random_state=SEED)
    samples.append(chosen)

sample = pd.concat(samples)
sample = sample.sort_values("article_id").reset_index(drop=True)

# Build the exact text that both models will use.
docs = []
for title, body in zip(sample["title"], sample["body"]):
    opening = " ".join(body.split()[:OPENING_WORDS])
    docs.append(title + "\n\n" + opening)

print("Articles in our sample:", len(docs))
print("First article IDs:", sample["article_id"].head().tolist())
print("\nFirst model input:\n", docs[0])

## 2. Turn text into word counts

### Link back to the lecture slides: the document–term matrix

Recall **“The document-term matrix from Week 4”** in the Week 5 slides, and the Week 4 example **“A document-term matrix with CountVectorizer”**. Rows represent documents, columns represent terms, and each cell contains a word count.

In the slides, `dtm = vectorizer.fit_transform(docs)` creates that sparse document–term matrix directly. Here we build the **same kind of bag-of-words representation**, in the format Gensim expects:

- `dictionary` defines the vocabulary and its word IDs: these play the role of column indices.
- Each entry in `corpus` represents one document: this plays the role of a row.
- `(word_id, count)` records a nonzero entry in that row. Zero counts are left out.

So `(7, 3)` means “the term with ID 7 occurs three times in this document”. You can look up the term with `dictionary[7]`. The vocabulary IDs may be ordered differently from scikit-learn's columns, but the idea is the same.

`build_analyzer()` only prepares the word-processing function; it does not create a matrix. The loop applies that function, and `doc2bow()` counts the resulting words. We use this format because the LDA implementation in this notebook comes from Gensim.

`CountVectorizer` splits text into words, lowercases them and removes common English stopwords. The analyzer lets us inspect the resulting tokens. This simple approach keeps single words; it does not understand meaning or distinguish different senses of a word.

Gensim needs two objects: a **dictionary** mapping words to IDs, and a **corpus** containing `(word_id, count)` pairs for each article. A pair `(7, 3)` means word 7 appears three times in that article. It is not a topic assignment.

We use the same tokenisation and reference counts in both notebooks. LDA fits these counts. BERTopic uses embeddings for grouping and uses counts to describe the groups. The reference counts will also be used for coherence.

We explicitly add “said”, “mr”, “mrs” and “ms” to the stopwords because they often describe news-writing style rather than the issue. This is a documented teaching choice, not a universal rule. Both notebooks use the same tokenisation and reference counts for coherence. BERTopic's encoder still reads the original text. These choices can change results; try changing one later.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from gensim.corpora import Dictionary

news_stopwords = list(ENGLISH_STOP_WORDS) + ["said", "mr", "mrs", "ms"]
vectorizer = CountVectorizer(stop_words=news_stopwords)
analyze = vectorizer.build_analyzer()

tokens = []
for document in docs:
    tokens.append(analyze(document))

dictionary = Dictionary(tokens)
corpus = []
for words in tokens:
    corpus.append(dictionary.doc2bow(words))

print("First tokens:", tokens[0][:20])
print("First word-ID/count pairs:", corpus[0][:10])
print("Vocabulary size:", len(dictionary))

## 3. Fit, compare and choose an LDA model

### 🟦 Model 1: the original count-based model

We start with five topics because this is a small classroom exercise. The editorial categories are not passed to the model. You can later try a different topic count and inspect what changes.

`passes=10` means ten passes through the corpus. The seed controls random initialisation. In the printed output, word weights describe the topic's word distribution; they are not article-topic shares. Displayed leading words show only part of the full distribution.

In [ ]:
from gensim.models import LdaModel

N_TOPICS = 5
lda_1 = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=N_TOPICS,
    passes=10,
    random_state=SEED,
)

for topic_id, words in lda_1.print_topics(num_topics=-1, num_words=5):
    print(topic_id, words)

### 🟧 Fit it again: change the number of topics

Before choosing a final model, try a small change. The first model is saved as `lda_1`. The cell below repeats the same fit with **7 topics** and saves it as `lda_2`.

**Change `num_topics=7` to another number and rerun this cell.** Keep the articles, seed and other settings fixed so you can see what changing K does. Read the printed topic words. Does the extra detail produce clearer themes or more mixed and repetitive topics?

In [ ]:
# lda_1 above remains the original model.

# CHANGE HERE: try a different number of topics.
lda_2 = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=7,
    passes=10,
    random_state=SEED,
)

for topic_id, words in lda_2.print_topics(num_topics=-1, num_words=5):
    print(topic_id, words)

### 🟩 Fit it again: add a few stopwords

This version goes back to the **original K** and changes only the stopwords. It is saved as `lda_3`.

**Edit `extra_stopwords` below and rerun the cell.** The example removes “year” and “new”, which were generic in some topics. Add words only if you can explain why they distract from your research question; removing a word can also remove useful information. We rebuild the dictionary and counts because their vocabulary changes.

This model still uses counts. The next version separately experiments with TF–IDF weights.

In [ ]:
# CHANGE HERE: choose additional stopwords based on your reading.
extra_stopwords = ["year", "new"]
new_vectorizer = CountVectorizer(stop_words=news_stopwords + extra_stopwords)
new_analyze = new_vectorizer.build_analyzer()

new_tokens = []
for document in docs:
    new_tokens.append(new_analyze(document))

new_dictionary = Dictionary(new_tokens)
new_corpus = []
for words in new_tokens:
    new_corpus.append(new_dictionary.doc2bow(words))

lda_3 = LdaModel(
    corpus=new_corpus,
    id2word=new_dictionary,
    num_topics=lda_1.num_topics,
    passes=10,
    random_state=SEED,
)

for topic_id, words in lda_3.print_topics(num_topics=-1, num_words=5):
    print(topic_id, words)

### 🟪 Model 4: fit LDA using TF–IDF weights

This version keeps the **original vocabulary, K, articles, seed and training passes**, but replaces token counts with TF–IDF weights. `TfidfModel` learns the weighting from our original corpus; applying it creates the weighted corpus for `lda_4`.

Standard LDA is designed for counts. Gensim accepts these fractional weights, but this changes the standard interpretation: treat this as an experiment, not an automatically better model. Compare its printed topic words and coherence with `lda_1`.

The common coherence reference remains the **original counts** for all four models. If you choose `lda_4`, the selection cell below also applies the fitted TF–IDF transformation when inferring article mixtures.

In [ ]:
from gensim.models import TfidfModel

tfidf_model = TfidfModel(corpus)
tfidf_corpus = list(tfidf_model[corpus])

lda_4 = LdaModel(
    corpus=tfidf_corpus,
    id2word=dictionary,
    num_topics=lda_1.num_topics,
    passes=10,
    random_state=SEED,
)

for topic_id, words in lda_4.print_topics(num_topics=-1, num_words=5):
    print(topic_id, words)

### Compare coherence before choosing

The table below compares the four models using **five leading words per topic and the same original reference counts**. This keeps the reference fixed even when a model removes stopwords. Rerun this cell whenever you change and refit `lda_2`, `lda_3` or `lda_4`.

**Higher `u_mass` is better: −2 is higher than −10.** Use the differences as one basis for your choice. There is no universal passing score, and a high score can come from generic words that frequently occur together. Also compare the printed topics: are they distinct and easy to name? Changing K also changes how many topic scores are averaged.

“Change from original” is the score minus the original model's score. A positive difference means higher coherence; a negative difference means lower coherence. Treat this as evidence to discuss alongside interpretability.

In [ ]:
from gensim.models import CoherenceModel

comparison_rows = []
for name, model in [("lda_1: original", lda_1),
                    ("lda_2: different K", lda_2),
                    ("lda_3: extra stopwords", lda_3),
                    ("lda_4: TF-IDF", lda_4)]:
    word_lists = []
    for topic_id in range(model.num_topics):
        words = [word for word, weight in model.show_topic(topic_id, topn=5)]
        word_lists.append(words)

    score = CoherenceModel(
        topics=word_lists, corpus=corpus, dictionary=dictionary,
        coherence="u_mass",
    ).get_coherence()
    comparison_rows.append({
        "Model": name,
        "Number of topics": model.num_topics,
        "Mean coherence (u_mass)": score,
    })

model_comparison = pd.DataFrame(comparison_rows)
baseline_score = model_comparison.loc[0, "Mean coherence (u_mass)"]
model_comparison["Change from original"] = (
    model_comparison["Mean coherence (u_mass)"] - baseline_score
)
model_comparison.round(3)

### 🟦 Choose the final model, then continue

**Change just the first line below to `lda = lda_1`, `lda = lda_2`, `lda = lda_3` or `lda = lda_4`.** Write down why you chose it: refer to the coherence difference and the topic words, not just the highest score.

Everything after this cell—article mixtures, reading, topic naming, the results DataFrame and visualisations—uses your selected `lda`. Its own dictionary supplies the correct word IDs. The original `corpus` and `dictionary` remain the shared reference for coherence.

Your choice is provisional until you read its articles. If they do not support the topics, change the selection and rerun **all cells from here down to the BONUS heading**. Topic IDs and names can change across models, so update the naming dictionary too.

For `lda_4`, article mixtures and the stability check use TF–IDF-weighted input. For the other models they use counts. The comparison and coherence evaluation always use the same original count reference.

In [ ]:
lda = lda_1  # CHANGE HERE: choose lda_1, lda_2, lda_3 or lda_4.

N_TOPICS = lda.num_topics
final_dictionary = lda.id2word
final_corpus = []
for words in tokens:
    final_corpus.append(final_dictionary.doc2bow(words))

# The TF-IDF model must also receive weighted input when inferring mixtures.
if lda is lda_4:
    final_corpus = list(tfidf_model[final_corpus])

# Refresh the leading words for the model you selected.
topic_words = {}
for topic_id in range(N_TOPICS):
    topic_words[topic_id] = [word for word, weight in lda.show_topic(topic_id, topn=5)]

for topic_id, words in lda.print_topics(num_topics=-1, num_words=5):
    print(topic_id, words)

## 4. Read topic mixtures and original articles

Use the final model selected above to infer a mixture for every excerpt. `minimum_probability=0` keeps every topic, including very small shares. Each row of the table represents one article; its shares sum to approximately 1. A share is a model estimate, not a count of manually labelled sentences.

In [ ]:
topic_shares = []
for article in final_corpus:
    mixture = lda.get_document_topics(article, minimum_probability=0)
    topic_shares.append([share for topic_id, share in mixture])

topic_shares = pd.DataFrame(topic_shares)
topic_shares.columns = [f"Topic {number}" for number in range(N_TOPICS)]
topic_shares.head()

Choose a topic ID below. Read three excerpts with high shares of it, propose a label, then look at other articles that might challenge it. To read a complete source article, use `print(sample.loc[article_id, "full_text"])`. Later paragraphs provide context but did not contribute to the model input.

In [ ]:
selected_topic = 0  # Change this to another topic ID.
print("Top words:", topic_words[selected_topic])

strongest_articles = topic_shares[f"Topic {selected_topic}"].nlargest(3)
for article_id, share in strongest_articles.items():
    print(f"\nArticle {article_id} | estimated topic share: {share:.0%}")
    print(docs[article_id])

## 5. Check coherence

Coherence asks whether the leading words tend to occur together in the original excerpts. We use Gensim's `u_mass` and five leading terms per topic, with the same reference counts in both notebooks.

**Higher is better: −2 is better than −10.** More strongly negative values mean weaker co-occurrence under this measure. Scores are commonly negative because the calculation uses log probabilities. There is no universal passing score. This is different from perplexity, where lower values indicate better predictive fit under comparable conditions.

The table and optional dot plot show individual topics, so a model-wide average does not hide weak ones. Points farther right have higher coherence. Read one lower-scoring and one higher-scoring topic before judging their usefulness. A high score does not establish that a human label fits.

[Gensim coherence documentation](https://radimrehurek.com/gensim/models/coherencemodel.html)

In [ ]:
from gensim.models import CoherenceModel

# topic_words holds the five leading words for each assigned topic.
coherence = CoherenceModel(
    topics=list(topic_words.values()),
    corpus=corpus,
    dictionary=dictionary,
    coherence="u_mass",
)

coherence_table = pd.DataFrame({
    "Topic": list(topic_words.keys()),
    "Top words": [", ".join(words) for words in topic_words.values()],
    "Coherence (u_mass)": coherence.get_coherence_per_topic(),
})
print("Mean coherence:", round(coherence_table["Coherence (u_mass)"].mean(), 3))
coherence_table

In [ ]:
import matplotlib.pyplot as plt

ordered = coherence_table.sort_values("Coherence (u_mass)")
plt.figure(figsize=(8, 4))
plt.scatter(ordered["Coherence (u_mass)"], range(len(ordered)))
plt.yticks(range(len(ordered)), ordered["Topic"])
plt.xlabel("Coherence (u_mass): higher is better →")
plt.ylabel("Topic ID")
plt.title("Which topics should we inspect more closely?")
plt.tight_layout()
plt.show()

## 6. Hands-on: name the topics and build your results DataFrame

**Your task:** inspect the topics, record your names in one editable dictionary, then attach those names to the articles. Complete this before the BONUS section.

1. Inspect the topic overview below. It lists each ID, leading words and three example article row numbers.
2. Read the example excerpts. Choose a short, specific name and note one supporting passage and one ambiguous example.
3. Edit the example names in the **Assign your names** dictionary below to match what you found. Keep the numeric topic IDs as keys. Give different IDs distinct names so summaries stay easy to interpret.
4. Run the mapping, DataFrame and plotting cells. If you revise a name, edit the same dictionary and rerun those cells.

This keeps all naming decisions visible and reproducible in the notebook, without interactive prompts. Name topics before looking at the original news categories. Names do not change fitted topics. After refitting, inspect the new IDs and revise the mapping rather than reusing old names automatically.

In [ ]:
topic_rows = []
for topic_id, words in topic_words.items():
    examples = topic_shares[f"Topic {topic_id}"].nlargest(3).index.tolist()
    topic_rows.append({
        "Topic ID": topic_id,
        "Leading words": ", ".join(words),
        "Example article rows": examples,
    })

topic_overview = pd.DataFrame(topic_rows)
display(topic_overview)

# Read the model input for the example articles.
for row in topic_rows:
    print(f"\nTOPIC {row['Topic ID']}: {row['Leading words']}")
    for article_id in row["Example article rows"]:
        print(f"\nArticle {article_id}: {docs[article_id]}")

### Assign your names in one place

Edit the example dictionary below directly. The number on the left is a topic ID; the text on the right is your name for it. **These are example names, not verified labels for your model.** Compare each ID with the overview and original excerpts, then replace its name. Add or remove entries if your model found a different number of topics.

After changing a name, rerun the mapping, DataFrame and plotting cells below. The mapping cell reports missing names or IDs that are not in your current model. BERTopic keeps `-1` as “Unassigned”.

In [ ]:
# Placeholder names only: replace these after reading your model's topics.
# IDs start at 0. Add or remove entries to match the topic overview above.
topic_names = {
    0: "Example topic 1",
    1: "Example topic 2",
    2: "Example topic 3",
    3: "Example topic 4",
    4: "Example topic 5",
}

In [ ]:
# Apply your names and report anything that still needs attention.
topic_labels = {}
for topic_id in topic_words:
    label = topic_names.get(topic_id, "").strip()
    if label == "":
        label = f"Topic {topic_id} (not named yet)"
        print(f"Please add a name for topic {topic_id} in topic_names.")
    topic_labels[topic_id] = label

for topic_id in topic_names:
    if topic_id not in topic_words:
        print(f"Topic {topic_id} is not in this model. Check your dictionary.")

topic_labels

### Attach the model output to the articles

`results` has **one row per article**, with its original ID, title, model excerpt and original category. We add the ID and name of the topic with the largest estimated share, that largest share, and all topic-share columns.

`argmax(axis=1)` finds the position of the largest share in each row. Here the topic columns are ordered 0, 1, 2, …, so this position is the topic ID. `map(topic_labels)` looks up your name for each ID. `concat(..., axis=1)` places the share columns alongside the metadata. Their row order matches because both were made from the same `sample`.

The largest-share topic is a convenient summary. **LDA still gives each article a mixture**, and an article with a largest share of 0.30 is quite different from one with a largest share of 0.95.

In [ ]:
results = sample[["article_id", "title", "category"]].copy()
results["text"] = docs
results["topic_id"] = topic_shares.values.argmax(axis=1)
results["topic_label"] = results["topic_id"].map(topic_labels)
results["largest_topic_share"] = topic_shares.max(axis=1)
results = pd.concat([results, topic_shares], axis=1)

results.head()

### Describe and plot your named topics

Count how many articles have each topic name and express that count as a percentage of the sample. `value_counts()` counts rows; dividing by `len(results)` gives a share of articles.

**Try it:** which named topic is largest? How many articles and what percentage does it contain? Choose one topic, read more of its articles using the filter below, and write two sentences describing what it covers and where its label falls short.

In [ ]:
topic_counts = results["topic_label"].value_counts()
summary = topic_counts.rename("Articles").to_frame()
summary["Percent of articles"] = (100 * summary["Articles"] / len(results)).round(1)
display(summary)

topic_counts.sort_values().plot.barh(figsize=(9, 5))
plt.xlabel("Number of articles")
plt.ylabel("Your topic name")
plt.title("Articles by named topic")
plt.tight_layout()
plt.show()

### LDA: describe the mixtures as well

The count plot above assigns an article to its largest-share topic for convenience. A second view averages **each topic's estimated share across all articles**, retaining the mixtures. These two summaries answer different questions and can give different rankings.

The chart below shows mean estimated topic shares, not percentages of articles manually coded into a category. Compare it with the count plot and explain one difference.

In [ ]:
mean_shares = topic_shares.mean()

# Replace the generic column names with your own topic names.
column_names = {}
for topic_id, label in topic_labels.items():
    column_names[f"Topic {topic_id}"] = label
mean_shares = mean_shares.rename(index=column_names)

mean_shares.sort_values().plot.barh(figsize=(9, 5))
plt.xlabel("Mean estimated topic share per article")
plt.ylabel("Your topic name")
plt.title("LDA topic mixtures across the sample")
plt.tight_layout()
plt.show()

In [ ]:
# Replace this with another name from your summary table.
label_to_read = results["topic_label"].iloc[0]
chosen_articles = results.loc[results["topic_label"] == label_to_read]
print("Articles with this label:", len(chosen_articles))

for article_id in chosen_articles.head(3).index:
    print(f"\n{results.loc[article_id, 'title']}")
    print(results.loc[article_id, "text"])

### Finish the hands-on task

Before continuing, make sure every assigned topic has your own name. Then write:

1. The name, count and percentage of the largest topic.
2. A short description of one topic, supported by an excerpt and an ambiguous example.
3. One limitation of the bar chart. For LDA, compare counts with mean topic shares; for BERTopic, report how many articles are unassigned.

Your `results` DataFrame is now ready for ordinary pandas filtering, grouping and plotting. If you want a CSV, uncomment the export below. It writes to your current notebook folder (the temporary session storage in Colab), not automatically to Google Drive. If you share it, include the BBC dataset attribution and source-use conditions from section 1.

In [ ]:
# Optional: save your labelled results for later use.
# results.to_csv("named_topic_results.csv", index=False)

## 7. Compare with the original news categories

Now that you have named the topics yourself, reveal the original five news categories as a reference. They were not supplied to the model. The table counts articles from each original category within each model group. A group can span categories, and a category can contain several meaningful subtopics.

Do not assume that a topic model must reproduce the five editorial categories exactly. Read examples to decide whether an unexpected grouping is useful. For LDA, this table uses each article's largest topic share as a simplified summary; the article still has a mixture.

In [ ]:
# Compare your own names with the original editorial categories.
pd.crosstab(results["topic_label"], results["category"])

## 8. More ways to inspect your results

Try these views after naming your topics. You do not have to use every plot. For each one, choose something you notice and return to the underlying articles to check your interpretation. Give the plots your own descriptive titles when reporting results.

### Category heatmap: what is inside each named topic?

Each row is one of your topic names; columns are the original BBC categories. Each row sums to 100%. A dark cell means many articles **within that topic** belong to that category. This normalisation makes differently sized topics easier to compare, but it hides their absolute sizes: use the count table alongside it.

This is a comparison with editorial categories, not a topic-similarity matrix. A mixed row might reveal a useful cross-category theme or a poorly separated topic. Read examples to distinguish these explanations. For LDA, the row uses each article's largest-share topic and does not show its full mixture.

In [ ]:
category_percentages = pd.crosstab(
    results["topic_label"], results["category"], normalize="index"
) * 100

display(category_percentages.round(1))
plt.figure(figsize=(10, 5))
plt.imshow(category_percentages, aspect="auto", cmap="Blues", vmin=0, vmax=100)
plt.xticks(range(len(category_percentages.columns)), category_percentages.columns,
           rotation=45, ha="right")
plt.yticks(range(len(category_percentages.index)), category_percentages.index)
plt.colorbar(label="Percent of articles within this topic")
plt.xlabel("Original news category")
plt.ylabel("Your topic name")
plt.title("Which news categories make up each topic?")
plt.tight_layout()
plt.show()

### Word weights: inspect one topic's vocabulary

Change `topic_to_plot` to inspect another topic. The bars show probabilities of words **within this topic**, not word counts or article-topic shares. The displayed words are only part of the topic's vocabulary, so their weights do not need to sum to 1.

Do these words support your topic name? Which additional words or original articles would you inspect?

In [ ]:
topic_to_plot = 0  # Choose a topic ID from your topic-label dictionary.
weighted_words = lda.show_topic(topic_to_plot, topn=10)
words = []
weights = []
for word, weight in weighted_words:
    words.append(word)
    weights.append(weight)

plt.figure(figsize=(8, 4))
plt.barh(words[::-1], weights[::-1])
plt.xlabel("Word probability within this topic")
plt.title(topic_labels[topic_to_plot])
plt.tight_layout()
plt.show()

### Largest-share distribution: how mixed are the articles?

This histogram counts articles by the size of their largest topic share. Values near 1 mean one topic dominates the estimated mixture; smaller values mean the mixture is more spread out. A dominant share is not proof that the model is correct, and a mixed article is not automatically a bad result.

In [ ]:
results["largest_topic_share"].plot.hist(bins=15, figsize=(8, 4))
plt.xlabel("Largest estimated topic share in an article")
plt.ylabel("Number of articles")
plt.title("Do articles mostly have one dominant topic?")
plt.tight_layout()
plt.show()

## Optional: stability across five random seeds

**Question:** would the same settings give similar themes after a fresh fit? We fit the selected model five times on the **same articles, vocabulary, input weights, K and training settings**, changing only its random seed. If you selected TF–IDF, `final_corpus` is already weighted and is reused unchanged.

Five runs are a practical classroom starting point, not a methodological standard. These test random-initialisation sensitivity, not sensitivity to collecting a different sample. The fits below are separate models and do not replace your selected `lda`, names or results.

Topic IDs can move between runs. We compare the five leading words and match topics **one-to-one** by maximising their total word overlap. The overlap for two word sets is shared words divided by all distinct words in the pair (Jaccard overlap). One means identical word sets; zero means no shared words. It ignores word order and weights and can be inflated by generic shared words. There is no universal good-score cut-off.

In [ ]:
stability_seeds = [42, 43, 44, 45, 46]
lda_stability_models = []

for seed in stability_seeds:
    repeated_lda = LdaModel(
        corpus=final_corpus,
        id2word=final_dictionary,
        num_topics=lda.num_topics,
        passes=lda.passes,
        iterations=lda.iterations,
        random_state=seed,
    )
    lda_stability_models.append(repeated_lda)
    print(f"\nSeed {seed}")
    for topic_id, words in repeated_lda.print_topics(num_topics=-1, num_words=5):
        print(topic_id, words)

### Compare every pair of runs

Five runs give ten pairs. For each pair we compute all topic-word overlaps. `linear_sum_assignment` chooses a one-to-one matching with the highest total overlap, so several topics cannot all select the same counterpart. The first table gives the mean matched overlap per pair; the second shows the weakest individual matches for closer reading.

The ten pairs share fitted models, so they are not ten independent experiments. The minimum and maximum describe observed variation; they are not a confidence interval.

In [ ]:
from itertools import combinations
from scipy.optimize import linear_sum_assignment

lda_pair_rows = []
lda_match_rows = []

for first_run, second_run in combinations(range(len(stability_seeds)), 2):
    first_model = lda_stability_models[first_run]
    second_model = lda_stability_models[second_run]
    first_topics = []
    second_topics = []
    for topic_id in range(lda.num_topics):
        first_topics.append([word for word, weight in first_model.show_topic(topic_id, topn=5)])
        second_topics.append([word for word, weight in second_model.show_topic(topic_id, topn=5)])

    overlaps = []
    for first_words in first_topics:
        row = []
        for second_words in second_topics:
            shared = set(first_words) & set(second_words)
            all_words = set(first_words) | set(second_words)
            row.append(len(shared) / len(all_words))
        overlaps.append(row)

    first_ids, second_ids = linear_sum_assignment(overlaps, maximize=True)
    matched_scores = []
    for first_id, second_id in zip(first_ids, second_ids):
        overlap = overlaps[first_id][second_id]
        matched_scores.append(overlap)
        lda_match_rows.append({
            "First seed": stability_seeds[first_run],
            "Second seed": stability_seeds[second_run],
            "First topic": first_id, "Second topic": second_id,
            "First words": ", ".join(first_topics[first_id]),
            "Second words": ", ".join(second_topics[second_id]),
            "Word overlap": overlap,
        })
    lda_pair_rows.append({
        "First seed": stability_seeds[first_run],
        "Second seed": stability_seeds[second_run],
        "Mean matched overlap": sum(matched_scores) / len(matched_scores),
    })

lda_stability_pairs = pd.DataFrame(lda_pair_rows)
lda_stability_matches = pd.DataFrame(lda_match_rows)
display(lda_stability_pairs.round(3))
display(lda_stability_pairs["Mean matched overlap"].agg(["median", "min", "max"]).round(3))
display(lda_stability_matches.sort_values("Word overlap").head(10).round(3))

### Example interpretation — one checked run, not a target score

> “Across five seeds, the median pairwise mean top-word overlap was approximately 0.37, ranging from 0.32 to 0.45. Leading vocabularies changed appreciably despite keeping the same K and input data. I would inspect the strongest and weakest matched word lists and read their articles before claiming that a particular theme was recovered consistently.”

For a **single** pair of five-word topics, an overlap near 0.67 corresponds to four shared words out of six distinct words. The model-pair means average several matched topics, so they need not have that direct interpretation.

**Write your own interpretation:** report the median and range, identify a strong and a weak match using the word lists, then read original excerpts from those themes. State whether your substantive conclusion would change. These vocabulary scores do not directly measure agreement between full article-topic mixtures. Stable generic topics can still be unhelpful; stability complements coherence and reading.

These numbers came from one checked run of the provided BBC exercise. Your selected model, package versions and computing environment can change the results. Use your own tables when writing your conclusion; these numbers are not acceptance thresholds.


## Optional: interactive topic map

Each circle is a topic; its size reflects estimated prevalence. The map is a two-dimensional approximation of differences between topic word distributions, not a plot of articles. Click a circle to inspect its terms.

At `lambda=1`, terms are ranked by within-topic probability. Lower values give more weight to distinctiveness. The map does not validate the topics. With `sort_topics=False`, circle 1 corresponds to model topic 0, circle 2 to topic 1, and so on.

If you selected `lda_4`, the map summarises the TF–IDF-weighted corpus; its frequency information reflects weights rather than original token counts. Interpret this exploratory model accordingly.

In [ ]:
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

visualisation = gensimvis.prepare(lda, final_corpus, final_dictionary, sort_topics=False)
pyLDAvis.display(visualisation)

## Something to think about

1. Propose a label for one topic and quote a passage supporting it. Find an ambiguous example too.
2. Compare a lower- and higher-coherence topic. Do your readings support that ranking?
3. Compare the model groups with the original news categories. What does the category table miss?
4. Compare LDA and BERTopic on the same article IDs. How do mixtures, groups and unassigned articles differ?

Both models use the same sample, excerpts and coherence reference. Their output structures and numbers of topics can still differ. Use several checks instead of choosing a winner from one score.

**Corpus-selection extension:** the earlier political-news archive produced many Trump-related terms. That can reveal a feature of the corpus; nearly identical topic descriptions can also indicate weak separation. Compare this broader-subject dataset with that narrower corpus. Removing names changes the measurement and does not remove the underlying subject.

These BBC files do not provide reliable per-article publication dates. We therefore do not plot topics over time or invent dates from filenames.

Compare your LDA versions: which change improved interpretability, and what evidence supports that claim? A higher coherence score alone is not enough.

## BONUS: the dated Canvas archive and topics over time

**BONUS — optional. Finish naming your topics, building `results` and making the plots above first.** The BBC exercise above is complete. Use this separate extension if you want to analyse the dated news in Canvas `articles.zip`. Keep the BBC results as your baseline; the variables below start with `archive_` so they do not replace it.

**Setup:** use the already extracted `articles` folder. There is no need to zip it again. In Colab, connect Drive below and set `ARCHIVE_PATH` to that folder. On your own computer, put `articles/` next to the notebook or edit the path. The folder should contain dated folders, then outlet folders, then article files. If the folder check prints `False`, copy the correct folder path from Colab’s Files panel and update it before continuing.

The archive contains dated folders and outlet subfolders. We select up to 100 files per outlet using the same seed, then model each title plus the first 100 body words. An equal-outlet sample is not representative of each outlet's publication volume. Filename dates are used as publication-date metadata; check the source documentation and a few original files before interpreting trends. Also check that filename titles match their bodies.

**Run the cells below only if you want to do this bonus and have the extracted `articles` folder ready. Otherwise, stop here.** If using Colab, uncomment the Drive connection lines below and authorize access; then set the folder path to its Drive location.

**Colab folder:** `My Drive/google-drive-colab/bdaca_week5/articles`, mounted as `/content/drive/MyDrive/google-drive-colab/bdaca_week5/articles`. Use the extracted folder.

If you open this local edition in Colab, uncomment the two Drive lines below. Use the `_colab` edition for ready-made installation and Drive cells.

In [ ]:
# Colab only (optional):
# from google.colab import drive
# drive.mount("/content/drive")

In [ ]:
from pathlib import Path

# Point to the extracted articles folder, not to articles.zip.
# Expected layout: articles/<dated folder>/<outlet>/<article file>.
# In Colab: /content/drive/MyDrive/google-drive-colab/bdaca_week5/articles
ARCHIVE_PATH = Path("articles")
print("Folder exists:", ARCHIVE_PATH.is_dir())

In [ ]:
import random

archive_outlets = ["BBC", "CNN", "The Guardian", "Vox", "Infowars"]
archive_rng = random.Random(SEED)
archive_rows = []

for outlet in archive_outlets:
    files = sorted(ARCHIVE_PATH.glob(f"*/{outlet}/*"))
    files = [file for file in files if file.is_file()]
    if not files:
        raise FileNotFoundError(
            f"No articles found for {outlet} under {ARCHIVE_PATH}. "
            "Set ARCHIVE_PATH to the extracted articles folder containing the dated folders."
        )
    chosen_files = archive_rng.sample(files, min(100, len(files)))
    for file in chosen_files:
        source, date, title = file.name.split("--", 2)
        body = file.read_text(encoding="utf-8").strip()
        opening = " ".join(body.split()[:OPENING_WORDS])
        archive_rows.append({
            "date": date, "outlet": outlet,
            "text": title + "\n\n" + opening,
        })

archive_data = pd.DataFrame(archive_rows)
archive_data["month"] = pd.to_datetime(archive_data["date"]).dt.to_period("M").dt.to_timestamp()
archive_docs = archive_data["text"].tolist()
print("Archive excerpts:", len(archive_docs))
display(pd.crosstab(archive_data["month"], archive_data["outlet"]))

### Fit a separate LDA model and average its shares by month

We fit one model on all sampled archive excerpts, then infer each article's topic mixture. We do not refit separately for each month: topic 0 keeps the same meaning throughout this analysis.

The chart shows **mean estimated topic share per sampled article**, not article counts or manually coded percentages. Every article has equal weight. Inspect the month-by-outlet table above: a changing mix of outlets can affect the trend. Months with no articles are absent, and a plotted line can bridge gaps. Read articles around an apparent change before connecting it to an event.

This is a simple topics-over-time summary, **not a dynamic LDA model whose topic-word distributions evolve over time**.

In [ ]:
archive_tokens = []
for document in archive_docs:
    archive_tokens.append(analyze(document))
archive_dictionary = Dictionary(archive_tokens)
archive_corpus = []
for words in archive_tokens:
    archive_corpus.append(archive_dictionary.doc2bow(words))

archive_lda = LdaModel(
    corpus=archive_corpus, id2word=archive_dictionary,
    num_topics=5, passes=10, random_state=SEED,
)
for topic_id, words in archive_lda.print_topics(num_topics=-1, num_words=5):
    print(topic_id, words)

archive_shares = []
for article in archive_corpus:
    mixture = archive_lda.get_document_topics(article, minimum_probability=0)
    archive_shares.append([share for topic_id, share in mixture])
archive_share_table = pd.DataFrame(archive_shares)
archive_share_table.columns = [f"Topic {number}" for number in range(5)]
archive_share_table["month"] = archive_data["month"]

archive_monthly = archive_share_table.groupby("month").mean()
archive_monthly.plot(figsize=(10, 5), marker="o")
plt.xlabel("Publication month")
plt.ylabel("Mean topic share per article opening")
plt.title("LDA topics in the sampled Canvas archive")
plt.tight_layout()
plt.show()

### Discuss the archive results

Do several topics mention Trump? Is the model separating issues such as elections, trade or immigration, or mainly repeating the same actor names? Corpus selection, date range and the sample shape what the model can find. Similar-looking topics require further reading and validation.

For a stricter comparison between LDA and BERTopic on this optional archive, keep sample settings and text construction identical and repeat the encoder-length check on `archive_docs`. The LDA and BERTopic charts above measure different things, so compare interpretations and coverage rather than the heights of their lines.